# 04 Report Visualizations

Generate paper-style report figures directly from experiment CSV files. This notebook computes every dataframe and plot in-place; it does not read previously generated PNG files.

## Setup

In [ ]:
from pathlib import Path
import json

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXPERIMENT_NAME = "compare_l100_i3d_siglip2_minprop_auto"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "report_figures" / EXPERIMENT_NAME
VIDEO_DIR = PROJECT_ROOT / "Charades" / "Charades_v1_480" / "Charades_v1_480"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METHOD_ORDER = ["baseline", "query_decomp", "qc_fr", "bu_pg", "full"]
METHOD_LABELS = {
    "baseline": "Baseline",
    "query_decomp": "Query Decomp.",
    "qc_fr": "QC-FR",
    "bu_pg": "BU-PG",
    "full": "Full",
}
METHOD_COLORS = {
    "baseline": "#3b6ea8",
    "query_decomp": "#4fa99f",
    "qc_fr": "#e69f42",
    "bu_pg": "#d95f5f",
    "full": "#6c5ce7",
}

# Paper-style report metrics only. We intentionally exclude IoU@0.1 and IoU@0.3 here.
PAPER_METRICS = ["R@1_IoU_0.5", "R@1_IoU_0.7", "mIoU"]
PAPER_METRIC_LABELS = {
    "R@1_IoU_0.5": "R@1 IoU=0.5",
    "R@1_IoU_0.7": "R@1 IoU=0.7",
    "mIoU": "mIoU",
}
SAMPLE_KEYS = ["video_id", "query", "gt_start", "gt_end"]

PROJECT_ROOT, EXPERIMENT_NAME, OUTPUT_DIR


## Load CSV Outputs

In [ ]:
def experiment_paths(project_root, experiment_name):
    root = Path(project_root)
    return {
        "metrics": root / "outputs" / "experiments" / f"{experiment_name}_metrics.csv",
        "predictions": root / "outputs" / "predictions" / experiment_name,
    }


def sort_methods(df):
    df = df.copy()
    df["method"] = pd.Categorical(df["method"], categories=METHOD_ORDER, ordered=True)
    return df.sort_values("method")


def load_metrics(project_root, experiment_name):
    path = experiment_paths(project_root, experiment_name)["metrics"]
    if not path.exists():
        raise FileNotFoundError(path)
    return sort_methods(pd.read_csv(path))


def load_predictions(project_root, experiment_name):
    prediction_dir = experiment_paths(project_root, experiment_name)["predictions"]
    if not prediction_dir.exists():
        raise FileNotFoundError(prediction_dir)

    paths = sorted(
        prediction_dir.glob("*.csv"),
        key=lambda path: METHOD_ORDER.index(path.stem) if path.stem in METHOD_ORDER else 99,
    )
    frames = []
    for path in paths:
        frame = pd.read_csv(path)
        if "method" not in frame or frame["method"].isna().all():
            frame["method"] = path.stem
        frames.append(frame)

    if not frames:
        raise FileNotFoundError(f"No prediction CSV files found in {prediction_dir}")

    df = pd.concat(frames, ignore_index=True)
    df["method_label"] = df["method"].map(METHOD_LABELS).fillna(df["method"])
    df["gt_length"] = df["gt_end"] - df["gt_start"]
    df["pred_length"] = df["pred_end"] - df["pred_start"]
    df["center_error"] = np.abs(
        0.5 * (df["pred_start"] + df["pred_end"]) - 0.5 * (df["gt_start"] + df["gt_end"])
    )
    return df


metrics = load_metrics(PROJECT_ROOT, EXPERIMENT_NAME)
predictions = load_predictions(PROJECT_ROOT, EXPERIMENT_NAME)
metrics.shape, predictions.shape


## Paper-Style Metric Table

In [ ]:
paper_table = metrics[["method"] + PAPER_METRICS].copy()
paper_table["method"] = paper_table["method"].map(METHOD_LABELS).fillna(paper_table["method"])
paper_table


## Helper Functions

In [ ]:
def method_label(method):
    return METHOD_LABELS.get(method, method)


def ordered_methods(df):
    available = set(df["method"])
    ordered = [method for method in METHOD_ORDER if method in available]
    extras = sorted(method for method in available if method not in METHOD_ORDER)
    return ordered + extras


def apply_axis_style(ax, grid_axis="y"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis=grid_axis, color="#d9d9d9", linewidth=0.8, alpha=0.75)
    ax.set_axisbelow(True)


def save_figure(fig, name):
    path = OUTPUT_DIR / name
    fig.savefig(path, dpi=260, bbox_inches="tight")
    print(path)
    return path


def prediction_wide(predictions_df):
    df = predictions_df.drop_duplicates(SAMPLE_KEYS + ["method"])
    wide = df.pivot_table(
        index=SAMPLE_KEYS,
        columns="method",
        values=["iou", "pred_start", "pred_end", "pred_length"],
        aggfunc="first",
    )
    wide.columns = [f"{value}_{method}" for value, method in wide.columns]
    wide = wide.reset_index()
    wide["gt_length"] = wide["gt_end"] - wide["gt_start"]
    return wide


def shorten(text, max_len=82):
    text = str(text)
    return text if len(text) <= max_len else text[: max_len - 3] + "..."


## Figure 1: Paper Metrics

In [ ]:
def plot_paper_metrics(metrics_df):
    df = sort_methods(metrics_df)
    methods = df["method"].tolist()
    labels = [method_label(method) for method in methods]
    x = np.arange(len(methods))
    width = 0.24
    metric_colors = ["#3b6ea8", "#d95f5f", "#4fa99f"]

    fig, ax = plt.subplots(figsize=(9.2, 4.8))
    apply_axis_style(ax)
    for index, metric in enumerate(PAPER_METRICS):
        values = pd.to_numeric(df[metric], errors="coerce").to_numpy(dtype=float)
        positions = x + (index - 1) * width
        bars = ax.bar(
            positions,
            values,
            width=width,
            label=PAPER_METRIC_LABELS[metric],
            color=metric_colors[index],
            alpha=0.84,
        )
        for bar, value in zip(bars, values):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.015,
                f"{value:.2f}",
                ha="center",
                va="bottom",
                fontsize=8,
            )

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=15, ha="right")
    ax.set_ylim(0, max(0.6, float(df[PAPER_METRICS].max().max()) + 0.12))
    ax.set_ylabel("Score")
    ax.set_title("Paper-Style Retrieval Metrics", weight="bold")
    ax.legend(ncol=3, frameon=False, loc="upper right")
    return fig


fig = plot_paper_metrics(metrics)
paper_metrics_path = save_figure(fig, "paper_metrics_comparison.png")
display(fig)
plt.close(fig)


## Figure 2: IoU Change Relative to Baseline

In [ ]:
def plot_iou_delta_vs_baseline(predictions_df):
    wide = prediction_wide(predictions_df)
    if "iou_baseline" not in wide.columns:
        raise ValueError("Baseline predictions are required for this figure.")

    methods = [method for method in METHOD_ORDER if method != "baseline" and f"iou_{method}" in wide.columns]
    deltas = [wide[f"iou_{method}"] - wide["iou_baseline"] for method in methods]
    labels = [method_label(method) for method in methods]

    fig, ax = plt.subplots(figsize=(8.8, 4.8))
    apply_axis_style(ax)
    box = ax.boxplot(deltas, labels=labels, patch_artist=True, showfliers=False, widths=0.55)
    for patch, method in zip(box["boxes"], methods):
        patch.set_facecolor(METHOD_COLORS.get(method, "#999999"))
        patch.set_alpha(0.68)
    for median in box["medians"]:
        median.set_color("#111111")
        median.set_linewidth(1.5)

    rng = np.random.default_rng(7)
    for index, (method, delta) in enumerate(zip(methods, deltas), start=1):
        jitter = rng.normal(loc=index, scale=0.035, size=len(delta))
        ax.scatter(jitter, delta, s=13, color=METHOD_COLORS.get(method, "#777777"), alpha=0.18, edgecolors="none")
        ax.scatter(index, float(delta.mean()), marker="D", s=42, color="#111111", zorder=4)
        ax.text(index, float(delta.mean()) + 0.035, f"mean {delta.mean():+.2f}", ha="center", fontsize=8)

    ax.axhline(0, color="#111111", linewidth=1.1, linestyle="--")
    ax.set_ylabel("IoU change vs. baseline")
    ax.set_title("Per-Query IoU Improvement Over Baseline", weight="bold")
    ax.tick_params(axis="x", rotation=15)
    return fig


fig = plot_iou_delta_vs_baseline(predictions)
iou_delta_path = save_figure(fig, "iou_delta_vs_baseline.png")
display(fig)
plt.close(fig)


## Figure 3: Predicted Segment Length

In [ ]:
def plot_length_alignment(predictions_df):
    methods = ordered_methods(predictions_df)
    unique_gt = predictions_df.drop_duplicates(SAMPLE_KEYS)["gt_length"]
    data = [unique_gt.to_numpy()]
    labels = ["Ground Truth"]

    for method in methods:
        rows = predictions_df[predictions_df["method"] == method]
        data.append(rows["pred_length"].to_numpy())
        labels.append(method_label(method))

    fig, ax = plt.subplots(figsize=(9.6, 4.9))
    apply_axis_style(ax)
    box = ax.boxplot(data, labels=labels, patch_artist=True, showfliers=False, widths=0.55)
    colors = ["#e45756"] + [METHOD_COLORS.get(method, "#999999") for method in methods]
    for patch, color in zip(box["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.62)
    for median in box["medians"]:
        median.set_color("#111111")
        median.set_linewidth(1.5)

    gt_median = float(unique_gt.median())
    ax.axhline(gt_median, color="#e45756", linestyle="--", linewidth=1.3, label=f"GT median: {gt_median:.2f}s")
    upper = np.nanpercentile(np.concatenate([np.asarray(series, dtype=float) for series in data]), 95) * 1.15
    ax.set_ylim(0, max(5.0, upper))
    ax.set_ylabel("Moment length (seconds)")
    ax.set_title("Predicted Length Calibration", weight="bold")
    ax.tick_params(axis="x", rotation=18)
    ax.legend(frameon=False, loc="upper right")
    return fig


fig = plot_length_alignment(predictions)
length_path = save_figure(fig, "length_alignment.png")
display(fig)
plt.close(fig)


## Figure 4: Qualitative Timelines

In [ ]:
def add_example(examples, row, reason, used_keys):
    key = tuple(row[column] for column in SAMPLE_KEYS)
    if key in used_keys:
        return False
    examples.append((row, reason))
    used_keys.add(key)
    return True


def best_delta_row(wide, method, used_keys, descending=True):
    delta = wide[f"iou_{method}"] - wide["iou_baseline"]
    ordered = delta.sort_values(ascending=not descending).index
    for index in ordered:
        row = wide.loc[index]
        key = tuple(row[column] for column in SAMPLE_KEYS)
        if key not in used_keys:
            return row, float(delta.loc[index])
    return None, None


def select_qualitative_examples(predictions_df, max_examples=2, min_best_iou=0.5):
    wide = prediction_wide(predictions_df)
    examples = []
    used_keys = set()
    iou_columns = [column for column in wide.columns if column.startswith("iou_")]
    wide["_best_iou"] = wide[iou_columns].max(axis=1)

    # Prefer examples where a proposed module clearly improves over baseline.
    for method in ["bu_pg", "full", "qc_fr"]:
        if "iou_baseline" not in wide.columns or f"iou_{method}" not in wide.columns:
            continue
        delta = wide[f"iou_{method}"] - wide["iou_baseline"]
        candidates = wide[(delta > 0) & (wide["_best_iou"] >= min_best_iou)].copy()
        candidates["_delta"] = delta.loc[candidates.index]
        for _, row in candidates.sort_values("_delta", ascending=False).iterrows():
            reason = f"{method_label(method)} improves over baseline ({row['_delta']:+.2f} IoU)"
            if add_example(examples, row, reason, used_keys):
                break
        if len(examples) >= max_examples:
            return examples[:max_examples]

    # Fill remaining slots with successful examples only; all-miss cases are not useful for the report figure.
    if len(examples) < max_examples:
        successful = wide[wide["_best_iou"] >= min_best_iou].sort_values("_best_iou", ascending=False)
        for _, row in successful.iterrows():
            reason = f"Representative successful case (best IoU {row['_best_iou']:.2f})"
            if add_example(examples, row, reason, used_keys):
                if len(examples) >= max_examples:
                    break

    if not examples:
        fallback = wide.sort_values("_best_iou", ascending=False).iloc[0]
        add_example(examples, fallback, f"Best available case (best IoU {fallback['_best_iou']:.2f})", used_keys)

    return examples[:max_examples]


def plot_qualitative_timelines(predictions_df, max_examples=2):
    examples = select_qualitative_examples(predictions_df, max_examples=max_examples)
    methods = ordered_methods(predictions_df)
    rows_per_example = 1 + len(methods)
    fig_height = max(5.2, len(examples) * (0.55 * rows_per_example + 1.0))
    fig, axes = plt.subplots(len(examples), 1, figsize=(11.5, fig_height), squeeze=False)
    axes = axes.ravel()

    for ax, (sample, reason) in zip(axes, examples):
        apply_axis_style(ax, grid_axis="x")
        ax.spines["left"].set_visible(False)
        labels = ["Ground Truth"] + [method_label(method) for method in methods]
        y_positions = np.arange(len(labels))[::-1]
        bar_height = 0.62

        gt_start = float(sample["gt_start"])
        gt_end = float(sample["gt_end"])
        ax.axvspan(gt_start, gt_end, color="#e45756", alpha=0.08)
        ax.broken_barh(
            [(gt_start, gt_end - gt_start)],
            (y_positions[0] - bar_height / 2, bar_height),
            facecolors="#e45756",
            alpha=0.88,
        )

        y_labels = ["Ground Truth"]
        max_time = gt_end
        for index, method in enumerate(methods, start=1):
            start_col = f"pred_start_{method}"
            end_col = f"pred_end_{method}"
            iou_col = f"iou_{method}"
            if start_col not in sample or pd.isna(sample[start_col]):
                y_labels.append(method_label(method))
                continue

            pred_start = float(sample[start_col])
            pred_end = float(sample[end_col])
            iou = float(sample[iou_col])
            max_time = max(max_time, pred_end)
            ax.broken_barh(
                [(pred_start, max(0.0, pred_end - pred_start))],
                (y_positions[index] - bar_height / 2, bar_height),
                facecolors=METHOD_COLORS.get(method, "#999999"),
                alpha=0.80,
            )
            y_labels.append(f"{method_label(method)}  IoU={iou:.2f}")

        ax.set_yticks(y_positions)
        ax.set_yticklabels(y_labels)
        ax.set_xlim(0, max_time * 1.08 + 0.5)
        ax.set_xlabel("Time (seconds)")
        ax.set_title(f"{sample['video_id']}: {shorten(sample['query'])}", loc="left", weight="bold")
        ax.text(1.0, 1.05, reason, transform=ax.transAxes, ha="right", va="bottom", fontsize=9, color="#555555")

    fig.tight_layout(h_pad=1.6)
    return fig


fig = plot_qualitative_timelines(predictions)
timeline_path = save_figure(fig, "qualitative_timelines.png")
display(fig)
plt.close(fig)


## Figure 5: Single-Sample Pipeline Walkthrough

In [ ]:
def sample_rows_from_wide(predictions_df, sample):
    mask = np.ones(len(predictions_df), dtype=bool)
    for column in SAMPLE_KEYS:
        mask &= predictions_df[column].eq(sample[column])
    rows = predictions_df[mask].copy()
    rows["method"] = pd.Categorical(rows["method"], categories=METHOD_ORDER, ordered=True)
    return rows.sort_values("method")


def parse_json_cell(value, default):
    if pd.isna(value) or value == "":
        return default
    try:
        return json.loads(value)
    except Exception:
        return default


def select_walkthrough_sample(predictions_df):
    wide = prediction_wide(predictions_df)
    iou_columns = [column for column in wide.columns if column.startswith("iou_")]
    wide["_best_iou"] = wide[iou_columns].max(axis=1)
    if "iou_baseline" in wide.columns and "iou_full" in wide.columns:
        wide["_walkthrough_score"] = wide["iou_full"] - wide["iou_baseline"]
        candidates = wide[(wide["iou_full"] >= 0.5) & (wide["_walkthrough_score"] > 0)]
        if len(candidates):
            return candidates.sort_values("_walkthrough_score", ascending=False).iloc[0]
    return wide.sort_values("_best_iou", ascending=False).iloc[0]


def draw_sample_timeline(ax, rows, sample):
    apply_axis_style(ax, grid_axis="x")
    ax.spines["left"].set_visible(False)
    labels = ["Ground Truth"] + [method_label(method) for method in rows["method"]]
    y_positions = np.arange(len(labels))[::-1]
    bar_height = 0.62
    gt_start = float(sample["gt_start"])
    gt_end = float(sample["gt_end"])
    max_time = gt_end

    ax.axvspan(gt_start, gt_end, color="#e45756", alpha=0.08)
    ax.broken_barh([(gt_start, gt_end - gt_start)], (y_positions[0] - bar_height / 2, bar_height), facecolors="#e45756", alpha=0.88)
    y_labels = ["Ground Truth"]
    for index, (_, row) in enumerate(rows.iterrows(), start=1):
        pred_start = float(row["pred_start"])
        pred_end = float(row["pred_end"])
        max_time = max(max_time, pred_end)
        method = row["method"]
        ax.broken_barh(
            [(pred_start, max(0.0, pred_end - pred_start))],
            (y_positions[index] - bar_height / 2, bar_height),
            facecolors=METHOD_COLORS.get(method, "#999999"),
            alpha=0.80,
        )
        y_labels.append(f"{method_label(method)}  IoU={row['iou']:.2f}")

    ax.set_yticks(y_positions)
    ax.set_yticklabels(y_labels)
    ax.set_xlim(0, max_time * 1.08 + 0.5)
    ax.set_xlabel("Time (seconds)")


def plot_sample_walkthrough(predictions_df, sample=None):
    sample = select_walkthrough_sample(predictions_df) if sample is None else sample
    rows = sample_rows_from_wide(predictions_df, sample)
    fig = plt.figure(figsize=(11.6, 8.2))
    grid = fig.add_gridspec(3, 2, height_ratios=[0.85, 1.85, 1.25], width_ratios=[1.25, 1.0], hspace=0.52, wspace=0.28)

    ax_info = fig.add_subplot(grid[0, 0])
    ax_info.axis("off")
    ax_info.text(0, 0.94, "Step 1: Input sample", fontsize=12, weight="bold")
    ax_info.text(0, 0.58, f"Video: {sample['video_id']}", fontsize=10)
    ax_info.text(0, 0.34, f"Query: {shorten(sample['query'], 72)}", fontsize=10)
    ax_info.text(0, 0.10, f"Ground truth: {sample['gt_start']:.2f}s - {sample['gt_end']:.2f}s", fontsize=10)

    ax_iou = fig.add_subplot(grid[0, 1])
    apply_axis_style(ax_iou, grid_axis="x")
    colors = [METHOD_COLORS.get(method, "#999999") for method in rows["method"]]
    ax_iou.barh([method_label(method) for method in rows["method"]], rows["iou"], color=colors, alpha=0.82)
    ax_iou.set_xlim(0, 1.0)
    ax_iou.set_xlabel("IoU")
    ax_iou.set_title("Step 3: Evaluation", weight="bold")

    ax_timeline = fig.add_subplot(grid[1, :])
    draw_sample_timeline(ax_timeline, rows, sample)
    ax_timeline.set_title("Step 2: Method predictions on the same timeline", weight="bold", loc="left")

    ax_diag = fig.add_subplot(grid[2, :])
    ax_diag.axis("off")
    proposal_rows = rows[rows["method"].isin(["qc_fr", "bu_pg", "full"])]
    table_rows = []
    for _, row in proposal_rows.iterrows():
        score = row.get("candidate_score", "")
        score_text = "-" if pd.isna(score) or score == "" else f"{float(score):.3f}"
        selected = parse_json_cell(row.get("selected_candidates", ""), [])
        selected_count = len(selected) if isinstance(selected, list) else 0
        table_rows.append([
            method_label(row["method"]),
            int(row.get("candidate_count", 0)),
            int(row.get("combination_count", 0)),
            int(row.get("overlap_candidate_count", 0)),
            selected_count,
            score_text,
            f"{row['pred_start']:.2f}-{row['pred_end']:.2f}s",
        ])
    ax_diag.text(0.0, 1.08, "Step 4: Proposal diagnostics", fontsize=12, weight="bold", transform=ax_diag.transAxes)
    table = ax_diag.table(
        cellText=table_rows,
        colLabels=["Method", "Candidates", "Combinations", "Overlap kept", "Selected", "Score", "Final span"],
        cellLoc="center",
        loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.4)
    fig.suptitle("Single-Sample Pipeline Walkthrough", fontsize=15, weight="bold", y=0.99)
    return fig


fig = plot_sample_walkthrough(predictions)
sample_walkthrough_path = save_figure(fig, "sample_walkthrough.png")
display(fig)
plt.close(fig)


## Figure 6: Video Frame Walkthrough

In [ ]:
def sample_times(start, end, count, duration):
    start = max(0.0, min(float(start), float(duration)))
    end = max(0.0, min(float(end), float(duration)))
    if end <= start:
        end = min(float(duration), start + 0.1)
    if count == 1:
        return [0.5 * (start + end)]
    return np.linspace(start, end, count + 2)[1:-1]


def read_frames(video_id, spans, frames_per_span=5):
    video_path = VIDEO_DIR / f"{video_id}.mp4"
    if not video_path.exists():
        raise FileNotFoundError(video_path)

    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if fps <= 0 or frame_total <= 0:
        cap.release()
        raise RuntimeError(f"Could not read video frames from {video_path}")

    duration = frame_total / fps
    frame_rows = []
    for label, start, end, color in spans:
        row = []
        for time in sample_times(start, end, frames_per_span, duration):
            frame_index = min(max(int(round(time * fps)), 0), frame_total - 1)
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
            ok, frame = cap.read()
            if not ok:
                frame = np.zeros((224, 224, 3), dtype=np.uint8)
            else:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            row.append((float(time), frame))
        frame_rows.append((label, color, row))
    cap.release()
    return frame_rows


def frame_spans(predictions_df, sample):
    rows = sample_rows_from_wide(predictions_df, sample)
    spans = [("Ground Truth", sample["gt_start"], sample["gt_end"], "#e45756")]
    for _, row in rows.iterrows():
        label = f"{method_label(row['method'])}\nIoU={row['iou']:.2f}"
        color = METHOD_COLORS.get(row["method"], "#999999")
        spans.append((label, row["pred_start"], row["pred_end"], color))
    return spans


def plot_video_frame_walkthrough(predictions_df, sample=None, frames_per_span=5):
    sample = select_walkthrough_sample(predictions_df) if sample is None else sample
    frame_rows = read_frames(sample["video_id"], frame_spans(predictions_df, sample), frames_per_span=frames_per_span)
    fig, axes = plt.subplots(len(frame_rows), frames_per_span, figsize=(frames_per_span * 3.0, len(frame_rows) * 2.15))
    if len(frame_rows) == 1:
        axes = np.asarray([axes])

    for row_index, (label, color, frames) in enumerate(frame_rows):
        for col_index, (time, frame) in enumerate(frames):
            ax = axes[row_index, col_index]
            ax.imshow(frame)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_title(f"{time:.1f}s", fontsize=10)
            for spine in ax.spines.values():
                spine.set_color(color)
                spine.set_linewidth(3)
            if col_index == 0:
                ax.set_ylabel(label, rotation=0, ha="right", va="center", fontsize=10, labelpad=34)

    fig.suptitle(
        f"Video Frame Walkthrough: {sample['video_id']} - {shorten(sample['query'], 70)}",
        fontsize=14,
        weight="bold",
        y=0.995,
    )
    fig.tight_layout(rect=[0.05, 0, 1, 0.97])
    return fig


fig = plot_video_frame_walkthrough(predictions)
video_frame_walkthrough_path = save_figure(fig, "video_frame_walkthrough.png")
display(fig)
plt.close(fig)


## Saved Figure Paths

In [ ]:
figure_paths = {
    "paper_metrics_comparison": paper_metrics_path,
    "iou_delta_vs_baseline": iou_delta_path,
    "length_alignment": length_path,
    "qualitative_timelines": timeline_path,
    "sample_walkthrough": sample_walkthrough_path,
    "video_frame_walkthrough": video_frame_walkthrough_path,
}
figure_paths
